# DATA 266 Lab 2 - Part 1: Multimodal RAG

End-to-end PDF ingestion, visual records, dense plus BM25 retrieval, conversation memory, grounded JSON answers, Kaggle submission format, evaluation, and safety checks. Put the provided Part1 folder beside this notebook or set PART1_DIR. Missing inputs are reported, not replaced with invented results.

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Please enable a GPU runtime.")

GPU available: True
GPU name: Tesla T4


In [2]:
%pip install -q pymupdf sentence-transformers rank-bm25 pandas pillow

In [3]:
import os
import re
import fitz
import pandas as pd
import numpy as np
import time

from pathlib import Path
from PIL import Image
from io import BytesIO

PART1_DIR = Path("/content")

print("Files inside /content:")
for file in sorted(PART1_DIR.iterdir()):
    print(file.name)

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Files inside /content:
.config
sample_data


In [5]:
from pathlib import Path
import shutil

required_files = [
    "Document.pdf",
    "Document2_Supplement_A_Outturns_2011-2013.pdf",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf",
    "Questions.csv",
    "kb_manifest.json"
]

for filename in required_files:
    source = Path("/") / filename
    destination = Path("/content") / filename

    if source.exists():
        shutil.copy2(source, destination)
        print("Copied:", filename)
    else:
        print("Not found:", source)

Copied: Document.pdf
Copied: Document2_Supplement_A_Outturns_2011-2013.pdf
Copied: Document3_Briefing_B_Labour_Prices_Policy.pdf
Copied: Questions.csv
Copied: kb_manifest.json


In [6]:
required_files = [
    "Document.pdf",
    "Document2_Supplement_A_Outturns_2011-2013.pdf",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf",
    "Questions.csv",
    "kb_manifest.json"
]

missing_files = [
    file for file in required_files
    if not (PART1_DIR / file).exists()
]

if missing_files:
    print("Missing files:")
    for file in missing_files:
        print("-", file)
else:
    print("All required Part 1 files are available.")

All required Part 1 files are available.


In [7]:
trust_levels = {
    "Document.pdf": "high",
    "Document2_Supplement_A_Outturns_2011-2013.pdf": "high",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf": "medium"
}

document_ids = {
    "Document.pdf": "DOC1",
    "Document2_Supplement_A_Outturns_2011-2013.pdf": "DOC2",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf": "DOC3"
}

pdf_files = [
    PART1_DIR / "Document.pdf",
    PART1_DIR / "Document2_Supplement_A_Outturns_2011-2013.pdf",
    PART1_DIR / "Document3_Briefing_B_Labour_Prices_Policy.pdf"
]

records = []
visual_records = []

for pdf_file in pdf_files:
    doc_id = document_ids[pdf_file.name]
    trust_level = trust_levels[pdf_file.name]

    pdf_document = fitz.open(pdf_file)

    print(f"Processing {pdf_file.name}...")

    for page_number, page in enumerate(pdf_document, start=1):

        page_text = page.get_text("text")

        # Create text chunks
        chunk_size = 1400

        for start in range(0, len(page_text), chunk_size):
            chunk = page_text[start:start + 1600].strip()

            if chunk:
                records.append({
                    "doc_id": doc_id,
                    "page": page_number,
                    "text": chunk,
                    "trust_level": trust_level,
                    "source": f"{doc_id}:p{page_number}"
                })

        # Convert each page into an image
        image_folder = PART1_DIR / "extracted_images"
        image_folder.mkdir(exist_ok=True)

        image_path = image_folder / f"{doc_id}_page_{page_number}.png"

        pixmap = page.get_pixmap(
            matrix=fitz.Matrix(1.5, 1.5),
            alpha=False
        )

        page_image = Image.open(
            BytesIO(pixmap.tobytes("png"))
        )

        page_image.save(image_path)

        # Detect figure and table labels
        labels = re.findall(
            r"(?i)\b(?:Figure|Fig\.?|Table)\s+[A-Z]?[-]?\d+[A-Za-z-]*",
            page_text
        )

        visual_records.append({
            "doc_id": doc_id,
            "page": page_number,
            "image_path": str(image_path),
            "labels": sorted(set(labels)),
            "caption": page_text[:500],
            "trust_level": trust_level,
            "source": f"{doc_id}:p{page_number}"
        })

print("\nPDF ingestion completed.")
print("Number of text chunks:", len(records))
print("Number of visual records:", len(visual_records))

Processing Document.pdf...
Processing Document2_Supplement_A_Outturns_2011-2013.pdf...
Processing Document3_Briefing_B_Labour_Prices_Policy.pdf...

PDF ingestion completed.
Number of text chunks: 129
Number of visual records: 51


In [8]:
print("Example text record:")
print(records[0])

print("\nExample visual record:")
print(visual_records[0])

Example text record:
{'doc_id': 'DOC1', 'page': 1, 'text': '4 \nIntroduction \nThe Core\n1-1 The Crisis\nTable 1-1 gives you output growth rates for the world economy, for advanced econ-\nomies and for other countries separately, since 2000. As you can see, from 2000 to \n2007 the world economy had a sustained expansion. Annual average world output \ngrowth was 3.2%, with advanced economies (the group of 30 or so richest countries \nin the world) growing at 2.6% per year, and emerging and developing economies \n(the other 150 or so other countries in the world) growing at an even faster 6.5% per \nyear.\nIn 2007 however, signs that the expansion might be coming to an end started to \nappear. U.S. housing prices, which had doubled since 2000, started declining. In mid-\n2007, as we wrote the previous edition of this book, we described how economists were \ndivided as to whether this might lead to a recession—a decrease in output. Optimists \nbelieved that, while lower housing prices mig

In [42]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import re
import numpy as np
import torch

# Extract document text
texts = [
    record["text"]
    for record in records
]

print("Total text chunks:", len(texts))

# Consistent tokenizer for BM25
TOKEN_RE = re.compile(
    r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?"
)

def tokenize(text):
    return TOKEN_RE.findall(
        str(text).lower()
    )

# Build BM25 index
tokenized_texts = [
    tokenize(text)
    for text in texts
]

bm25_index = BM25Okapi(
    tokenized_texts
)

# Select available device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Embedding device:", device)

# Load dense embedding model
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

# Generate normalized document embeddings
document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

print(
    "Embedding matrix shape:",
    document_embeddings.shape
)

print("Improved BM25 and dense indexes created successfully.")

Total text chunks: 129
Embedding device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding matrix shape: (129, 384)
Improved BM25 and dense indexes created successfully.


In [43]:
def dense_retrieve(
    query,
    top_k=5
):
    """
    Retrieves documents using dense semantic similarity.
    Supports expanded query variations.
    """

    query_variants = expanded_queries(
        query
    )

    all_scores = []

    for variant in query_variants:

        query_embedding = embedding_model.encode(
            [variant],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]

        scores = (
            document_embeddings @ query_embedding
        )

        all_scores.append(scores)

    # Keep the strongest semantic score for each document
    similarity_scores = np.max(
        np.stack(all_scores),
        axis=0
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = []

    for index in top_indices:

        result = records[index].copy()

        result["dense_score"] = float(
            similarity_scores[index]
        )

        results.append(result)

    return results

In [44]:
def bm25_retrieve(
    query,
    top_k=5
):
    """
    Retrieves documents using BM25 with
    expanded query variations.
    """

    query_variants = expanded_queries(
        query
    )

    all_scores = []

    for variant in query_variants:

        query_tokens = tokenize(
            variant
        )

        scores = bm25_index.get_scores(
            query_tokens
        )

        all_scores.append(
            np.asarray(scores)
        )

    # Keep the strongest keyword score for each document
    bm25_scores = np.max(
        np.stack(all_scores),
        axis=0
    )

    top_indices = np.argsort(
        bm25_scores
    )[::-1][:top_k]

    results = []

    for index in top_indices:

        result = records[index].copy()

        result["bm25_score"] = float(
            bm25_scores[index]
        )

        results.append(result)

    return results

In [45]:
def normalize_scores(scores):
    scores = np.asarray(
        scores,
        dtype=float
    )

    minimum = scores.min()
    maximum = scores.max()

    if maximum == minimum:
        return np.zeros_like(scores)

    return (
        scores - minimum
    ) / (
        maximum - minimum
    )


def hybrid_retrieve(
    query,
    top_k=8,
    dense_weight=0.55,
    bm25_weight=0.45
):
    """
    Combines dense retrieval and BM25 retrieval
    using expanded query variations.
    """

    query_variants = expanded_queries(
        query
    )

    dense_scores_all = []
    bm25_scores_all = []

    for variant in query_variants:

        # Dense semantic scores
        query_embedding = embedding_model.encode(
            [variant],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]

        dense_scores = (
            document_embeddings @ query_embedding
        )

        # BM25 keyword scores
        bm25_scores = bm25_index.get_scores(
            tokenize(variant)
        )

        dense_scores_all.append(
            np.asarray(dense_scores)
        )

        bm25_scores_all.append(
            np.asarray(bm25_scores)
        )

    # Keep the strongest score from all query variations
    dense_scores = np.max(
        np.stack(dense_scores_all),
        axis=0
    )

    bm25_scores = np.max(
        np.stack(bm25_scores_all),
        axis=0
    )

    # Normalize both retrieval methods
    normalized_dense = normalize_scores(
        dense_scores
    )

    normalized_bm25 = normalize_scores(
        bm25_scores
    )

    # Weighted hybrid score
    hybrid_scores = (
        dense_weight * normalized_dense
        + bm25_weight * normalized_bm25
    )

    # Add a small exact-keyword bonus
    query_terms = set(
        tokenize(query)
    )

    keyword_bonus = np.array([
        0.02 * len(
            query_terms.intersection(
                set(tokenize(record["text"]))
            )
        ) / max(len(query_terms), 1)
        for record in records
    ])

    hybrid_scores = (
        hybrid_scores + keyword_bonus
    )

    # Rank highest-scoring chunks
    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:top_k]

    results = []

    for index in top_indices:

        result = records[index].copy()

        result["dense_score"] = float(
            dense_scores[index]
        )

        result["bm25_score"] = float(
            bm25_scores[index]
        )

        result["hybrid_score"] = float(
            hybrid_scores[index]
        )

        results.append(result)

    return results

In [46]:
test_question = "What happened to labour prices?"

results = hybrid_retrieve(
    test_question,
    top_k=8
)

for rank, result in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Source:", result["source"])
    print("Document:", result["doc_id"])
    print("Page:", result["page"])
    print("Trust level:", result["trust_level"])
    print("Hybrid score:", round(result["hybrid_score"], 4))
    print("Text:", result["text"][:700])

Rank: 1
Source: DOC3:p1
Document: DOC3
Page: 1
Trust level: medium
Hybrid score: 0.9379
Text: e than either the GDP deflator or core CPI, mostly because of
energy prices. In 2009 the gap between headline and core CPI inflation was unusually large.
3. Participation in the U.S. labour force fell during the crisis, so the unemployment rate understates the
weakness of the labour market.
4. China's growth has slowed, but its registered urban unemployment rate has not moved.
Rank: 2
Source: DOC3:p1
Document: DOC3
Page: 1
Trust level: medium
Hybrid score: 0.8831
Text: DOC3  |  Briefing B: Labour Markets, Prices and Policy Trade-offs
Page 1
Briefing B
Labour Markets, Prices and Policy Trade-offs
after the Crisis
Document ID
DOC3
Publisher
Pacific Policy Institute (policy research group)
Date
30 June 2014
Source classification
Policy briefing (secondary analysis). Sections B.1-B.4 and the Erratum.
Trust level
Medium for Sections B.1-B.4 and the Erratum. Low for Appendix C (unverified reader
com

In [50]:
def normalize_scores(scores):
    scores = np.asarray(
        scores,
        dtype=float
    )

    minimum = scores.min()
    maximum = scores.max()

    if maximum == minimum:
        return np.zeros_like(scores)

    return (
        scores - minimum
    ) / (
        maximum - minimum
    )


def expanded_queries(query):
    query = str(query).strip()

    variants = [query]

    replacements = {
        "labour": ["labor", "labour market"],
        "prices": ["price", "costs", "inflation"],
        "wages": ["wage", "earnings", "pay"],
        "employment": ["jobs", "workers", "unemployment"],
        "output": ["production", "outturn"],
        "government": ["policy", "public sector"]
    }

    query_lower = query.lower()

    for word, alternatives in replacements.items():
        if word in query_lower:
            for alternative in alternatives:
                variants.append(
                    query_lower.replace(
                        word,
                        alternative
                    )
                )

    return list(
        dict.fromkeys(variants)
    )


def hybrid_retrieve(
    query,
    top_k=8,
    dense_weight=0.55,
    bm25_weight=0.45
):
    """
    Combines dense semantic retrieval
    and BM25 keyword retrieval.
    """

    query_variants = expanded_queries(
        query
    )

    dense_scores_all = []
    bm25_scores_all = []

    for variant in query_variants:

        query_embedding = embedding_model.encode(
            [variant],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]

        dense_scores = (
            document_embeddings @ query_embedding
        )

        bm25_scores = np.asarray(
            bm25_index.get_scores(
                tokenize(variant)
            )
        )

        dense_scores_all.append(
            dense_scores
        )

        bm25_scores_all.append(
            bm25_scores
        )

    # Keep the strongest score from all query variations
    dense_scores = np.max(
        np.stack(dense_scores_all),
        axis=0
    )

    bm25_scores = np.max(
        np.stack(bm25_scores_all),
        axis=0
    )

    normalized_dense = normalize_scores(
        dense_scores
    )

    normalized_bm25 = normalize_scores(
        bm25_scores
    )

    hybrid_scores = (
        dense_weight * normalized_dense
        + bm25_weight * normalized_bm25
    )

    # Exact keyword-match bonus
    query_terms = set(
        tokenize(query)
    )

    keyword_bonus = np.array([
        0.02 * len(
            query_terms.intersection(
                set(
                    tokenize(record["text"])
                )
            )
        ) / max(len(query_terms), 1)
        for record in records
    ])

    hybrid_scores = (
        hybrid_scores + keyword_bonus
    )

    # Rank chunks by final hybrid score
    ranked_indices = np.argsort(
        hybrid_scores
    )[::-1]

    results = []
    seen_sources = set()

    for index in ranked_indices:

        result = records[index].copy()
        source = result.get("source")

        # Keep one chunk per document page
        if source in seen_sources:
            continue

        seen_sources.add(source)

        result["dense_score"] = float(
            dense_scores[index]
        )

        result["bm25_score"] = float(
            bm25_scores[index]
        )

        result["hybrid_score"] = float(
            hybrid_scores[index]
        )

        results.append(result)

        if len(results) >= top_k:
            break

    return results

In [51]:
test_question = "What happened to labour prices?"

results = hybrid_retrieve(
    test_question,
    top_k=8
)

for rank, result in enumerate(
    results,
    start=1
):
    print("=" * 80)
    print("Rank:", rank)
    print("Source:", result["source"])
    print("Document:", result["doc_id"])
    print("Page:", result["page"])
    print("Hybrid score:", round(
        result["hybrid_score"],
        4
    ))
    print("Text:", result["text"][:500])

Rank: 1
Source: DOC3:p1
Document: DOC3
Page: 1
Hybrid score: 0.9379
Text: e than either the GDP deflator or core CPI, mostly because of
energy prices. In 2009 the gap between headline and core CPI inflation was unusually large.
3. Participation in the U.S. labour force fell during the crisis, so the unemployment rate understates the
weakness of the labour market.
4. China's growth has slowed, but its registered urban unemployment rate has not moved.
Rank: 2
Source: DOC3:p5
Document: DOC3
Page: 5
Hybrid score: 0.831
Text: DOC3  |  Briefing B: Labour Markets, Prices and Policy Trade-offs
Page 5
B.3 U.S. labour market indicators
Table B-2 reports our calculations of the U.S. unemployment and participation rates as annual
averages from household survey data. Participation fell by 1.3 percentage points between 2007 and
2010. Some workers who lost their jobs stopped looking for work and therefore are not counted as
unemployed. Long-term unemployment also rose sharply: mean duration increased

In [63]:
conversation_memory = {}

LABEL_PATTERN = re.compile(
    r"(?i)\b(?:Figure|Fig\.?|Table)\s+"
    r"[A-Z]?(?:-\d+|\d+(?:-\d+)?)"
)


def is_chart_noise(sentence):
    """
    Filters sentences that mostly contain
    chart-axis numbers or visual artifacts.
    """

    tokens = sentence.split()

    if len(tokens) < 8:
        return False

    numeric_tokens = sum(
        token.replace(".", "")
        .replace("-", "")
        .isdigit()
        for token in tokens
    )

    numeric_ratio = (
        numeric_tokens / len(tokens)
    )

    return numeric_ratio > 0.45


def answer_question(
    question,
    conversation_id="default",
    top_k=8
):
    """
    Returns a concise, grounded answer with
    citations and figure/table references.
    """

    question = str(
        question
    ).strip()

    # Handle session-only overrides
    override_pattern = (
        r"(?i)\b(assume|suppose|use)\b.*"
        r"\b(value|rate|number|price|growth|inflation)\b"
    )

    if re.search(
        override_pattern,
        question
    ):
        conversation_memory.setdefault(
            conversation_id,
            []
        ).append(question)

        return {
            "answer": (
                "The user-provided value was recorded "
                "as a conversation-only session override."
            ),
            "figure_references": [],
            "citations": [],
            "status": "session_override"
        }

    # Retrieve evidence
    retrieved_results = hybrid_retrieve(
        question,
        top_k=top_k
    )

    if not retrieved_results:
        return {
            "answer": (
                "The requested information was not found "
                "in the knowledge base."
            ),
            "figure_references": [],
            "citations": [],
            "status": "not_found"
        }

    # Use expanded query terms
    expanded_text = " ".join(
        expanded_queries(question)
    )

    question_terms = set(
        tokenize(expanded_text)
    )

    selected_sentences = []
    selected_sources = []
    seen_sentences = set()

    # Select relevant sentences
    for result in retrieved_results:

        text = result.get(
            "text",
            ""
        ).replace(
            "\n",
            " "
        ).strip()

        if not text:
            continue

        sentences = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        ranked_sentences = sorted(
            sentences,
            key=lambda sentence: len(
                question_terms.intersection(
                    set(
                        tokenize(sentence)
                    )
                )
            ),
            reverse=True
        )

        for sentence in ranked_sentences:

            sentence = sentence.strip()

            if (
                len(sentence) >= 25
                and sentence not in seen_sentences
                and not is_chart_noise(sentence)
            ):
                selected_sentences.append(
                    sentence
                )

                selected_sources.append(
                    result.get("source")
                )

                seen_sentences.add(
                    sentence
                )

            if len(selected_sentences) >= 5:
                break

        if len(selected_sentences) >= 5:
            break

    # Fallback if sentence filtering removes everything
    if not selected_sentences:

        fallback_text = retrieved_results[0].get(
            "text",
            ""
        ).replace(
            "\n",
            " "
        ).strip()

        selected_sentences = [
            fallback_text[:1200]
        ]

        selected_sources = [
            retrieved_results[0].get(
                "source"
            )
        ]

    # Create concise answer
    answer_text = " ".join(
        selected_sentences[:5]
    ).strip()

    answer_text = answer_text[:1200]

    # Extract valid figure/table references
    figure_references = []

    for result in retrieved_results[:5]:

        text_labels = LABEL_PATTERN.findall(
            result.get(
                "text",
                ""
            )
        )

        record_labels = [
            label.strip()
            for label in result.get(
                "labels",
                []
            )
            if LABEL_PATTERN.fullmatch(
                label.strip()
            )
        ]

        figure_references.extend(
            text_labels
        )

        figure_references.extend(
            record_labels
        )

    figure_references = list(
        dict.fromkeys(
            label.strip()
            for label in figure_references
            if label.strip()
        )
    )

    # Cite only pages that contributed selected sentences
    citations = list(
        dict.fromkeys(
            source
            for source in selected_sources
            if source
        )
    )

    # Fallback citations if sentence sources are unavailable
    if not citations:
        citations = list(
            dict.fromkeys(
                result.get("source")
                for result in retrieved_results[:3]
                if result.get("source")
            )
        )

    return {
        "answer": answer_text,
        "figure_references": figure_references,
        "citations": citations,
        "status": "answered"
    }

In [66]:
TOKEN_RE = re.compile(
    r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?"
)

def tokenize(text):
    return TOKEN_RE.findall(
        str(text).lower()
    )

print("Tokenizer is ready.")

Tokenizer is ready.


In [65]:
result = answer_question(
    "What happened to labour prices?",
    conversation_id="C001",
    top_k=8
)

print("ANSWER:")
print(result["answer"])

print("\nFIGURE REFERENCES:")
print(result["figure_references"])

print("\nCITATIONS:")
print(result["citations"])

print("\nSTATUS:")
print(result["status"])

ANSWER:
labour force fell during the crisis, so the unemployment rate understates the weakness of the labour market. e than either the GDP deflator or core CPI, mostly because of energy prices. In 2009 the gap between headline and core CPI inflation was unusually large. Participation in the U.S. China's growth has slowed, but its registered urban unemployment rate has not moved.

FIGURE REFERENCES:
['Table B-2', 'Figure B-5', 'Table 1-3', 'Figure C-1', 'Figure 2-4', 'Figure B-3', 'Figure B-4']

CITATIONS:
['DOC3:p1']

STATUS:
answered


In [67]:
questions_path = Path("/content/Questions.csv")

questions_df = pd.read_csv(
    questions_path,
    dtype=str,
    keep_default_na=False
)

print("Number of questions:", len(questions_df))
print("Columns:")
print(list(questions_df.columns))

display(questions_df.head())

Number of questions: 40
Columns:
['id', 'conversation_id', 'turn', 'question_id', 'question']


,id,conversation_id,turn,question_id,question
0,1,C001,1,Q001,What sparked the global economic crisis around...
1,2,C002,1,Q002,Why should we worry about unemployment rates g...
2,3,C003,1,Q003,How do economists measure economic growth with...
3,4,C004,1,Q004,How bad did the world economy get hit during t...
4,5,C005,1,Q005,What happened to U.S. unemployment after the 2...


In [68]:
# Clean column names
questions_df.columns = [
    str(column).strip()
    for column in questions_df.columns
]

# Find the question column
question_candidates = [
    "question",
    "Question",
    "question_text",
    "text",
    "prompt"
]

question_column = next(
    (
        column
        for column in question_candidates
        if column in questions_df.columns
    ),
    None
)

if question_column is None:
    raise ValueError(
        "Could not find the question column. "
        f"Available columns: {list(questions_df.columns)}"
    )

# Find the conversation column
conversation_candidates = [
    "conversation_id",
    "conversation",
    "conv_id",
    "conversationId"
]

conversation_column = next(
    (
        column
        for column in conversation_candidates
        if column in questions_df.columns
    ),
    None
)

if conversation_column is None:
    raise ValueError(
        "Could not find the conversation column. "
        f"Available columns: {list(questions_df.columns)}"
    )

print("Question column:", question_column)
print("Conversation column:", conversation_column)

Question column: question
Conversation column: conversation_id


In [69]:
submission_rows = []

start_time = time.perf_counter()

for _, row in questions_df.iterrows():

    question = row[question_column]

    if conversation_column is not None:
        conversation_id = row[conversation_column]
    else:
        conversation_id = str(row.get("id", "default"))

    result = answer_question(
        question=question,
        conversation_id=conversation_id,
        top_k=5
    )

    submission_rows.append({
        "id": row.get("id", ""),
        "conversation_id": row.get("conversation_id", ""),
        "question_id": row.get("question_id", ""),
        "answer": result["answer"],
        "figure_references": (
            ",".join(result["figure_references"])
            if result["figure_references"]
            else "NONE"
        ),
        "citations": (
            ",".join(result["citations"])
            if result["citations"]
            else "NONE"
        ),
        "status": result["status"]
    })

submission_df = pd.DataFrame(
    submission_rows,
    columns=[
        "id",
        "conversation_id",
        "question_id",
        "answer",
        "figure_references",
        "citations",
        "status"
    ]
)

submission_path = "/content/submission.csv"
submission_df.to_csv(
    submission_path,
    index=False
)

elapsed_time = time.perf_counter() - start_time

print("Submission created successfully.")
print("Rows:", len(submission_df))
print("Time:", round(elapsed_time, 2), "seconds")
display(submission_df.head())

Submission created successfully.
Rows: 40
Time: 2.47 seconds


,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,One might have hoped that the crisis would rem...,"Table 1-1,Figure 1-1,Figure 1-2,Figure 1-4,Tab...",DOC1:p2,answered
1,2,C002,Q002,unemployment rate has fluctuated between 3 and...,"Figure 2-3,Figure 1-4,Table 1-2",DOC1:p24,answered
2,3,C003,Q003,■ Economists care about unemployment because o...,Figure 2-4,DOC1:p33,answered
3,4,C004,Q004,Chapter 1 A Tour of the World 5 at risk of g...,"Figure 1-1,Table 1-1,Figure 2-2,Figure 2-1",DOC1:p2,answered
4,5,C005,Q005,8 Introduction The Core Then the crisis came...,"Figure 1-4,Table 1-2,Figure 1-1,Table 1-1,Figu...",DOC1:p5,answered


In [70]:
required_columns = [
    "id",
    "conversation_id",
    "question_id",
    "answer",
    "figure_references",
    "citations",
    "status"
]

submission_df = pd.read_csv(
    "/content/submission.csv",
    dtype=str,
    keep_default_na=False
)

print("Submission columns:")
print(list(submission_df.columns))

print("\nNumber of submission rows:", len(submission_df))
print("Number of question rows:", len(questions_df))

assert list(submission_df.columns) == required_columns, \
    "Submission columns are incorrect."

assert len(submission_df) == len(questions_df), \
    "Submission row count does not match Questions.csv."

print("\nBasic submission format is correct.")

Submission columns:
['id', 'conversation_id', 'question_id', 'answer', 'figure_references', 'citations', 'status']

Number of submission rows: 40
Number of question rows: 40

Basic submission format is correct.


In [71]:
valid_statuses = {
    "answered",
    "not_found",
    "conflict",
    "false_premise",
    "session_override"
}

invalid_statuses = set(
    submission_df["status"]
) - valid_statuses

print("Statuses found:")
print(submission_df["status"].value_counts())

if invalid_statuses:
    print("Invalid statuses:", invalid_statuses)
else:
    print("All statuses are valid.")

Statuses found:
status
answered    40
Name: count, dtype: int64
All statuses are valid.


In [72]:
empty_answers = (
    submission_df["answer"].str.strip() == ""
).sum()

empty_statuses = (
    submission_df["status"].str.strip() == ""
).sum()

print("Empty answers:", empty_answers)
print("Empty statuses:", empty_statuses)

if empty_answers == 0 and empty_statuses == 0:
    print("No required answer or status fields are empty.")

Empty answers: 0
Empty statuses: 0
No required answer or status fields are empty.


In [73]:
conversation_memory = {}

override_result = answer_question(
    "Assume the inflation rate is 5 percent.",
    conversation_id="TEST_CONVERSATION"
)

print(override_result)
print("\nConversation memory:")
print(conversation_memory)

{'answer': 'The user-provided value was recorded as a conversation-only session override.', 'figure_references': [], 'citations': [], 'status': 'session_override'}

Conversation memory:
{'TEST_CONVERSATION': ['Assume the inflation rate is 5 percent.']}


In [74]:
print(
    answer_question(
        "What happened to labour prices?",
        conversation_id="DIFFERENT_CONVERSATION"
    )
)

{'answer': "labour force fell during the crisis, so the unemployment rate understates the weakness of the labour market. e than either the GDP deflator or core CPI, mostly because of energy prices. In 2009 the gap between headline and core CPI inflation was unusually large. Participation in the U.S. China's growth has slowed, but its registered urban unemployment rate has not moved.", 'figure_references': ['Table B-2', 'Figure B-5', 'Table 1-3', 'Figure C-1', 'Figure 2-4', 'Figure B-3', 'Figure B-4'], 'citations': ['DOC3:p1'], 'status': 'answered'}


In [75]:
from google.colab import files

files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>